In [ ]:
!pip install scikeras -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, Flatten, LSTM, BatchNormalization, Input
from scikeras.wrappers import KerasClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
try:
    drive.mount('/content/drive')
except:
    print("Drive is already mounted.")


file_path = '/content/drive/MyDrive/DMT/dataset_synthetic_sonar.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Dataset loaded successfully from: {file_path}")
    print("Dataset shape:", df.shape)
    print("\nFirst 5 rows of the dataset:")
    print(df.head())
except FileNotFoundError:
    print(f"ERROR: File not found at the specified path: {file_path}")
    print("Please make sure the file exists and the path is correct.")


In [ ]:
X = df.iloc[:, 0:60].values
y = df['Label'].values

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
class_mapping = {label: index for index, label in enumerate(encoder.classes_)}
print(f"\nLabel Encoding Mapping: {class_mapping}")

In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\nData split complete:")
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")


In [ ]:
def evaluate_model(model_name, y_true, y_pred):
    """Prints classification report and plots a confusion matrix."""
    print(f"\n--- Evaluation for: {model_name} ---")
    print("Classification Report:")
    print(classification_report(y_true, y_pred, target_names=encoder.classes_))

    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.4f}")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=encoder.classes_, yticklabels=encoder.classes_)
    plt.title(f'Confusion Matrix for {model_name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()
    return acc

results = {}

In [ ]:
print("\n\n" + "="*50)
print("          TRAINING MACHINE LEARNING MODELS")
print("="*50)

In [ ]:
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
results['Logistic Regression'] = evaluate_model('Logistic Regression', y_test, y_pred_lr)

svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
results['SVM'] = evaluate_model('Support Vector Machine (SVM)', y_test, y_pred_svm)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
results['Random Forest'] = evaluate_model('Random Forest', y_test, y_pred_rf)

X_train_dl = np.expand_dims(X_train, axis=2)
X_test_dl = np.expand_dims(X_test, axis=2)

In [ ]:
print("\n\n" + "="*50)
print("          TRAINING DEEP LEARNING MODELS")
print("="*50)

In [ ]:
def create_cnn_model():
    model = Sequential([
        Input(shape=(60,1)),
        Conv1D(filters=32, kernel_size=3, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

cnn_model = create_cnn_model()
cnn_model.fit(X_train_dl, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.1)
y_pred_cnn_prob = cnn_model.predict(X_test_dl)
y_pred_cnn = (y_pred_cnn_prob > 0.5).astype(int)
results['CNN'] = evaluate_model('Convolutional Neural Network (CNN)', y_test, y_pred_cnn)

In [ ]:
def create_lstm_model():
    model = Sequential([
        Input(shape=(60,1)),
        LSTM(units=64, return_sequences=True),
        Dropout(0.3),
        LSTM(units=32),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

lstm_model = create_lstm_model()
lstm_model.fit(X_train_dl, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.1)
y_pred_lstm_prob = lstm_model.predict(X_test_dl)
y_pred_lstm = (y_pred_lstm_prob > 0.5).astype(int)
results['LSTM'] = evaluate_model('Long Short-Term Memory (LSTM)', y_test, y_pred_lstm)


In [ ]:
print("\n\n" + "="*50)
print("          TRAINING HYBRID STACKING ENSEMBLE")
print("="*50)

In [ ]:
cnn_clf = KerasClassifier(model=create_cnn_model, epochs=50, batch_size=32, verbose=0)
setattr(cnn_clf, '_estimator_type', 'classifier')

lstm_clf = KerasClassifier(model=create_lstm_model, epochs=50, batch_size=32, verbose=0)
setattr(lstm_clf, '_estimator_type', 'classifier')


base_estimators = [
    ('random_forest', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42)),
    ('cnn', cnn_clf),
    ('lstm', lstm_clf)
]

meta_learner = LogisticRegression(solver='liblinear')

stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=3,
    passthrough=True
)

print("Training the complete stacking model... (This may take a few minutes)")
stacking_model.fit(X_train, y_train)
print("Training complete!")

y_pred_stack = stacking_model.predict(X_test)
results['Hybrid Stacking Ensemble'] = evaluate_model('Hybrid Stacking Ensemble', y_test, y_pred_stack)

In [ ]:
print("\n\n" + "="*60)
print("          FINAL MODEL PERFORMANCE COMPARISON")
print("="*60)

In [ ]:
results_df = pd.DataFrame(
    list(results.items()),
    columns=['Model', 'Accuracy']
)

results_df = results_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

results_df['Accuracy'] = (results_df['Accuracy'] * 100).map('{:.2f}%'.format)


In [ ]:
print(results_df)

plt.figure(figsize=(12, 6))
ax = sns.barplot(x='Accuracy', y='Model', data=results_df, palette='viridis')
plt.title('Final Model Performance Comparison', fontsize=16)
plt.xlabel('Accuracy (%)', fontsize=12)
plt.ylabel('Model', fontsize=12)
for p in ax.patches:
    width = p.get_width()
    plt.text(width + 0.5, p.get_y() + p.get_height()/2. + 0.2,
             f'{width:.2f}%', ha='left', va='center')
plt.xlim(0, 105)
plt.show()

print("\nâœ… Project execution complete!")

In [ ]:
!pip install scikeras -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, Flatten, LSTM, BatchNormalization, Input
from scikeras.wrappers import KerasClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
try:
    drive.mount('/content/drive')
except:
    print("Drive is already mounted.")

print("\nâœ… Libraries imported and Google Drive mounted successfully!")


file_path = '/content/drive/MyDrive/DMT/dataset_synthetic_sonar.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Dataset loaded successfully from: {file_path}")
    print("Dataset shape:", df.shape)
except FileNotFoundError:
    print(f"ERROR: File not found at the specified path: {file_path}")
    df = pd.DataFrame()

In [ ]:







if not df.empty:
    X = df.iloc[:, 0:60].values
    y = df['Label'].values

    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y)

    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )

    print(f"\nData split complete:")
    print(f"Training data shape: {X_train.shape}")
    print(f"Testing data shape: {X_test.shape}")

    def evaluate_model(model_name, y_true, y_pred):
        print(f"\n--- Evaluation for: {model_name} ---")
        print("Classification Report:")
        print(classification_report(y_true, y_pred, target_names=encoder.classes_))
        acc = accuracy_score(y_true, y_pred)
        print(f"Accuracy: {acc:.4f}")
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=encoder.classes_, yticklabels=encoder.classes_)
        plt.title(f'Confusion Matrix for {model_name}')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.show()
        return acc

    results = {}

    print("\n\n" + "="*50)
    print("          TRAINING MACHINE LEARNING MODELS")
    print("="*50)
    lr_model = LogisticRegression(random_state=42)
    lr_model.fit(X_train, y_train)
    y_pred_lr = lr_model.predict(X_test)
    results['Logistic Regression'] = evaluate_model('Logistic Regression', y_test, y_pred_lr)

    svm_model = SVC(kernel='rbf', probability=True, random_state=42)
    svm_model.fit(X_train, y_train)
    y_pred_svm = svm_model.predict(X_test)
    results['SVM'] = evaluate_model('Support Vector Machine (SVM)', y_test, y_pred_svm)

    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_test)
    results['Random Forest'] = evaluate_model('Random Forest', y_test, y_pred_rf)

    X_train_dl = np.expand_dims(X_train, axis=2)
    X_test_dl = np.expand_dims(X_test, axis=2)

    print("\n\n" + "="*50)
    print("          TRAINING DEEP LEARNING MODELS")
    print("="*50)
    def create_cnn_model():
        model = Sequential([
            Input(shape=(60,1)),
            Conv1D(filters=32, kernel_size=3, activation='relu'),
            BatchNormalization(),
            MaxPooling1D(pool_size=2),
            Dropout(0.3),
            Flatten(),
            Dense(64, activation='relu'),
            Dense(1, activation='sigmoid')
        ])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model

    cnn_model = create_cnn_model()
    cnn_model.fit(X_train_dl, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.1)
    y_pred_cnn_prob = cnn_model.predict(X_test_dl)
    y_pred_cnn = (y_pred_cnn_prob > 0.5).astype(int)
    results['CNN'] = evaluate_model('Convolutional Neural Network (CNN)', y_test, y_pred_cnn)

    def create_lstm_model():
        model = Sequential([
            Input(shape=(60,1)),
            LSTM(units=64, return_sequences=True),
            Dropout(0.3),
            LSTM(units=32),
            Dropout(0.3),
            Dense(32, activation='relu'),
            Dense(1, activation='sigmoid')
        ])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model

    lstm_model = create_lstm_model()
    lstm_model.fit(X_train_dl, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.1)
    y_pred_lstm_prob = lstm_model.predict(X_test_dl)
    y_pred_lstm = (y_pred_lstm_prob > 0.5).astype(int)
    results['LSTM'] = evaluate_model('Long Short-Term Memory (LSTM)', y_test, y_pred_lstm)


    print("\n\n" + "="*50)
    print("          TRAINING HYBRID STACKING ENSEMBLE")
    print("="*50)

    cnn_clf = KerasClassifier(model=create_cnn_model, epochs=50, batch_size=32, verbose=0)
    lstm_clf = KerasClassifier(model=create_lstm_model, epochs=50, batch_size=32, verbose=0)

    setattr(cnn_clf, '_estimator_type', 'classifier')
    setattr(lstm_clf, '_estimator_type', 'classifier')

    base_estimators = [
        ('random_forest', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('svm', SVC(kernel='rbf', probability=True, random_state=42)),
        ('cnn', cnn_clf),
        ('lstm', lstm_clf)
    ]

    meta_learner = LogisticRegression(solver='liblinear')

    stacking_model = StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta_learner,
        cv=3,
        passthrough=True
    )

    print("Training the complete stacking model... (This may take a few minutes)")
    stacking_model.fit(X_train, y_train)
    print("Training complete!")

    y_pred_stack = stacking_model.predict(X_test)
    results['Hybrid Stacking Ensemble'] = evaluate_model('Hybrid Stacking Ensemble', y_test, y_pred_stack)

    print("\n\n" + "="*60)
    print("          FINAL MODEL PERFORMANCE COMPARISON")
    print("="*60)

    results_df = pd.DataFrame(
        list(results.items()),
        columns=['Model', 'Accuracy']
    ).sort_values(by='Accuracy', ascending=False)

    plt.figure(figsize=(12, 8))
    ax = sns.barplot(x=results_df['Accuracy']*100, y=results_df['Model'], palette='viridis', orient='h')
    plt.title('Final Model Performance Comparison', fontsize=16)
    plt.xlabel('Accuracy (%)', fontsize=12)
    plt.ylabel('Model', fontsize=12)
    plt.xlim(0, 105)
    for p in ax.patches:
        width = p.get_width()
        plt.text(width + 1, p.get_y() + p.get_height() / 2., f'{width:.2f}%', ha='left', va='center')
    plt.show()

    print("\nâœ… Project execution complete!")

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, MaxPooling1D, Dropout, Flatten, Dense, LSTM
from tensorflow.keras.optimizers import Adam

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
DATA_PATH = '/content/drive/MyDrive/DMT/dataset_synthetic_sonar.csv'
N_SPLITS = 5
RANDOM_STATE = 42
CNN_EPOCHS = 25
LSTM_EPOCHS = 25
BATCH_SIZE = 32

In [ ]:
df = pd.read_csv(DATA_PATH)
X = df.iloc[:, 0:60].values
y_raw = df['Label'].values

le = LabelEncoder()
y = le.fit_transform(y_raw)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

In [ ]:
def build_cnn():
    m = Sequential([
        Input(shape=(60,1)),
        Conv1D(32, 3, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return m

In [ ]:
def build_lstm():
    m = Sequential([
        Input(shape=(60,1)),
        LSTM(64, return_sequences=True),
        Dropout(0.3),
        LSTM(32),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return m


In [ ]:
n_train = X_train.shape[0]
n_test = X_test.shape[0]

meta_oof = np.zeros((n_train, 4))
meta_test = np.zeros((n_test, 4))

meta_test_fold_preds = np.zeros((N_SPLITS, n_test, 4))

rf_clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
svm_clf = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
    rf.fit(X_tr, y_tr)
    oof_rf = rf.predict_proba(X_val)[:,1]
    meta_oof[val_idx, 0] = oof_rf
    meta_test_fold_preds[fold, :, 0] = rf.predict_proba(X_test)[:,1]

    svm = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
    svm.fit(X_tr, y_tr)
    oof_svm = svm.predict_proba(X_val)[:,1]
    meta_oof[val_idx, 1] = oof_svm
    meta_test_fold_preds[fold, :, 1] = svm.predict_proba(X_test)[:,1]

    X_tr_c = np.expand_dims(X_tr, axis=2)
    X_val_c = np.expand_dims(X_val, axis=2)
    X_test_c = np.expand_dims(X_test, axis=2)

    cnn = build_cnn()
    cnn.fit(X_tr_c, y_tr, epochs=CNN_EPOCHS, batch_size=BATCH_SIZE, verbose=0, validation_data=(X_val_c, y_val))
    oof_cnn = cnn.predict(X_val_c, verbose=0).ravel()
    meta_oof[val_idx, 2] = oof_cnn
    meta_test_fold_preds[fold, :, 2] = cnn.predict(X_test_c, verbose=0).ravel()

    lstm = build_lstm()
    lstm.fit(X_tr_c, y_tr, epochs=LSTM_EPOCHS, batch_size=BATCH_SIZE, verbose=0, validation_data=(X_val_c, y_val))
    oof_lstm = lstm.predict(X_val_c, verbose=0).ravel()
    meta_oof[val_idx, 3] = oof_lstm
    meta_test_fold_preds[fold, :, 3] = lstm.predict(X_test_c, verbose=0).ravel()

    tf.keras.backend.clear_session()

meta_test = meta_test_fold_preds.mean(axis=0)

meta_clf = LogisticRegression(solver='liblinear', random_state=RANDOM_STATE)
meta_clf.fit(meta_oof, y_train)

meta_preds_proba = meta_clf.predict_proba(meta_test)[:,1]
meta_preds = (meta_preds_proba > 0.5).astype(int)

In [ ]:
print("\n=== Hybrid stacking (manual) evaluation ===")
print(classification_report(y_test, meta_preds, target_names=le.classes_))
acc = accuracy_score(y_test, meta_preds)
print(f"Stacked Accuracy: {acc*100:.2f}%")

cm = confusion_matrix(y_test, meta_preds)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix - Manual Stacking")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

### ------------------------------------------- ###
###      5. ENSEMBLE MODEL - MANUAL STACKING    ###
### ------------------------------------------- ###

Now, we will implement the stacking ensemble manually. This involves:
1. Training the base models on the training data.
2. Generating out-of-fold predictions from the base models on the training data to use as features for the meta-learner.
3. Training the meta-learner (Logistic Regression) on these out-of-fold predictions.
4. Generating predictions from the base models on the test data.
5. Using the test predictions from base models as input to the trained meta-learner to get the final ensemble predictions.

In [ ]:
from sklearn.model_selection import cross_val_predict

print("\n\n" + "="*50)
print("          MANUAL HYBRID STACKING ENSEMBLE")
print("="*50)



print("Generating out-of-fold predictions from base models on training data...")

X_train_lr = X_train
y_train_pred_lr = lr_model.predict_proba(X_train_lr)[:, 1]

X_train_svm = X_train
y_train_pred_svm = svm_model.predict_proba(X_train_svm)[:, 1]

X_train_rf = X_train
y_train_pred_rf = rf_model.predict_proba(X_train_rf)[:, 1]

X_train_cnn = np.expand_dims(X_train, axis=2)
y_train_pred_cnn = cnn_model.predict(X_train_cnn).flatten()

X_train_lstm = np.expand_dims(X_train, axis=2)
y_train_pred_lstm = lstm_model.predict(X_train_lstm).flatten()

X_meta_train = np.column_stack((y_train_pred_lr,
                                y_train_pred_svm,
                                y_train_pred_rf,
                                y_train_pred_cnn,
                                y_train_pred_lstm))

print(f"Shape of meta-training features: {X_meta_train.shape}")

meta_learner_model = LogisticRegression(solver='liblinear')
print("\nTraining the meta-learner...")
meta_learner_model.fit(X_meta_train, y_train)
print("Meta-learner training complete!")

print("\nGenerating predictions from base models on test data...")

X_test_lr = X_test
y_test_pred_lr = lr_model.predict_proba(X_test_lr)[:, 1]

X_test_svm = X_test
y_test_pred_svm = svm_model.predict_proba(X_test_svm)[:, 1]

X_test_rf = X_test
y_test_pred_rf = rf_model.predict_proba(X_test_rf)[:, 1]

X_test_cnn = np.expand_dims(X_test, axis=2)
y_test_pred_cnn = cnn_model.predict(X_test_cnn).flatten()

X_test_lstm = np.expand_dims(X_test, axis=2)
y_test_pred_lstm = lstm_model.predict(X_test_lstm).flatten()

X_meta_test = np.column_stack((y_test_pred_lr,
                               y_test_pred_svm,
                               y_test_pred_rf,
                               y_test_pred_cnn,
                               y_test_pred_lstm))

print(f"Shape of meta-testing features: {X_meta_test.shape}")

print("\nMaking final predictions with the meta-learner...")
y_pred_stack_manual = meta_learner_model.predict(X_meta_test)
print("Final predictions generated!")

results['Manual Stacking Ensemble'] = evaluate_model('Manual Stacking Ensemble', y_test, y_pred_stack_manual)

### ------------------------------------------- ###
###         6. PERFORMANCE EVALUATION           ###
### ------------------------------------------- ###

Now, let's update the final comparison table and plot with the results from the manual stacking ensemble.

In [ ]:
print("\n\n" + "="*60)
print("          FINAL MODEL PERFORMANCE COMPARISON")
print("="*60)

results_df = pd.DataFrame(
    list(results.items()),
    columns=['Model', 'Accuracy']
)

print_df = results_df.copy()
print_df['Accuracy'] = (print_df['Accuracy'] * 100).map('{:.2f}%'.format)
print("--- Model Accuracy Table ---")
print(print_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True))


results_df_sorted = results_df.sort_values(by='Accuracy', ascending=False)

plt.figure(figsize=(12, 8))
ax = sns.barplot(
    x=results_df_sorted['Accuracy'] * 100,
    y=results_df_sorted['Model'],
    palette='viridis',
    orient='h'
)

plt.title('Final Model Performance Comparison', fontsize=16)
plt.xlabel('Accuracy (%)', fontsize=12)
plt.ylabel('Model', fontsize=12)
plt.xlim(0, 105)

for p in ax.patches:
    width = p.get_width()
    plt.text(width + 1,
             p.get_y() + p.get_height() / 2.,
             f'{width:.2f}%',
             ha='left',
             va='center')

plt.show()

print("\nâœ… Project execution complete!")

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns


def get_all_metrics(y_true, y_pred):
    """Calculates accuracy, precision, recall, and F1-score."""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    return {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

all_results = {
    'Logistic Regression': get_all_metrics(y_test, y_pred_lr),
    'SVM': get_all_metrics(y_test, y_pred_svm),
    'Random Forest': get_all_metrics(y_test, y_pred_rf),
    'CNN': get_all_metrics(y_test, y_pred_cnn.flatten()),
    'LSTM': get_all_metrics(y_test, y_pred_lstm.flatten()),
    'Manual Stacking Ensemble': get_all_metrics(y_test, y_pred_stack_manual)
}

print("\n\n" + "="*60)
print("          DETAILED MODEL EVALUATION METRICS TABLE")
print("="*60)

metrics_df = pd.DataFrame.from_dict(all_results, orient='index')

metrics_df = metrics_df.sort_values(by='Accuracy', ascending=False)

print(metrics_df.to_string(formatters={
    'Accuracy': '{:,.2%}'.format,
    'Precision': '{:,.2%}'.format,
    'Recall': '{:,.2%}'.format,
    'F1-Score': '{:,.2%}'.format
}))


print("\n\n" + "="*60)
print("          VISUAL MODEL PERFORMANCE COMPARISON (ACCURACY)")
print("="*60)

plot_df = metrics_df.reset_index().rename(columns={'index': 'Model'})

plt.figure(figsize=(12, 8))
ax = sns.barplot(
    x='Accuracy',
    y='Model',
    data=plot_df,
    palette='viridis',
    orient='h'
)

plt.title('Final Model Performance Comparison', fontsize=16)
plt.xlabel('Accuracy', fontsize=12)
plt.ylabel('Model', fontsize=12)
plt.xlim(0, 1.05)

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

for p in ax.patches:
    width = p.get_width()
    plt.text(width + 0.01,
             p.get_y() + p.get_height() / 2.,
             f'{width:.2%}',
             ha='left',
             va='center')

plt.show()

print("\nâœ… Project execution complete!")